# Web Scraping

---
## 1. Dependencias

In [1]:
!pip install requests beautifulsoup4 pandas lxml tqdm openpyxl -q

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import logging
from urllib.parse import urljoin, urlparse
from tqdm.notebook import tqdm
from datetime import datetime
from pathlib import Path

logging.basicConfig(level=logging.WARNING)
Path('resultados').mkdir(exist_ok=True)

---
## 2. Lista das 69 Universidades Federais

In [ ]:
UNIVERSIDADES = [
    # Norte
    {"sigla": "UFAC",  "nome": "Universidade Federal do Acre",                         "url": "https://www.ufac.br"},
    {"sigla": "UNIFAP","nome": "Universidade Federal do Amapa",                        "url": "https://www.unifap.br"},
    {"sigla": "UFAM",  "nome": "Universidade Federal do Amazonas",                     "url": "https://www.ufam.edu.br"},
    {"sigla": "UFPA",  "nome": "Universidade Federal do Para",                         "url": "https://www.ufpa.br"},
    {"sigla": "UFRA",  "nome": "Universidade Federal Rural da Amazonia",               "url": "https://www.ufra.edu.br"},
    {"sigla": "UFRR",  "nome": "Universidade Federal de Roraima",                      "url": "https://www.ufrr.br"},
    {"sigla": "UFRO",  "nome": "Universidade Federal de Rondonia",                     "url": "https://www.unir.br"},
    {"sigla": "UFOPA", "nome": "Universidade Federal do Oeste do Para",                "url": "https://www.ufopa.edu.br"},
    {"sigla": "UNIFESSPA","nome": "Universidade Federal do Sul e Sudeste do Para",     "url": "https://www.unifesspa.edu.br"},
    {"sigla": "UFNT",  "nome": "Universidade Federal do Norte do Tocantins",           "url": "https://www.ufnt.edu.br"},
    {"sigla": "UFT",   "nome": "Universidade Federal do Tocantins",                    "url": "https://ww2.uft.edu.br"},

    # Nordeste
    {"sigla": "UFAL",  "nome": "Universidade Federal de Alagoas",                      "url": "https://ufal.br"},
    {"sigla": "UFRB",  "nome": "Universidade Federal do Reconcavo da Bahia",           "url": "https://www.ufrb.edu.br"},
    {"sigla": "UFBA",  "nome": "Universidade Federal da Bahia",                        "url": "https://www.ufba.br"},
    {"sigla": "UFSB",  "nome": "Universidade Federal do Sul da Bahia",                 "url": "https://ufsb.edu.br"},
    {"sigla": "UNIVASF","nome": "Universidade Federal do Vale do Sao Francisco",       "url": "https://www.univasf.edu.br"},
    {"sigla": "UFC",   "nome": "Universidade Federal do Ceara",                        "url": "https://www.ufc.br"},
    {"sigla": "UFCA",  "nome": "Universidade Federal do Cariri",                       "url": "https://www.ufca.edu.br"},
    {"sigla": "UNILAB","nome": "Universidade da Integracao Internacional da Lusofonia Afro-Brasileira", "url": "https://unilab.edu.br"},
    {"sigla": "UFMA",  "nome": "Universidade Federal do Maranhao",                     "url": "https://portais.ufma.br"},
    {"sigla": "UFCG",  "nome": "Universidade Federal de Campina Grande",               "url": "https://www.ufcg.edu.br"},
    {"sigla": "UFPB",  "nome": "Universidade Federal da Paraiba",                      "url": "https://www.ufpb.br"},
    {"sigla": "UFRPE", "nome": "Universidade Federal Rural de Pernambuco",             "url": "https://www.ufrpe.br"},
    {"sigla": "UFPE",  "nome": "Universidade Federal de Pernambuco",                   "url": "https://www.ufpe.br"},
    {"sigla": "UNIVASF","nome": "Universidade Federal do Vale do Sao Francisco",       "url": "https://www.univasf.edu.br"},
    {"sigla": "UFERSA","nome": "Universidade Federal Rural do Semi-Arido",             "url": "https://ufersa.edu.br"},
    {"sigla": "UFRN",  "nome": "Universidade Federal do Rio Grande do Norte",          "url": "https://www.ufrn.br"},
    {"sigla": "UFPI",  "nome": "Universidade Federal do Piaui",                        "url": "https://www.ufpi.br"},
    {"sigla": "UFS",   "nome": "Universidade Federal de Sergipe",                      "url": "https://www.ufs.br"},

    # Centro-Oeste
    {"sigla": "UFGD",  "nome": "Universidade Federal da Grande Dourados",              "url": "https://www.ufgd.edu.br"},
    {"sigla": "UFMS",  "nome": "Universidade Federal de Mato Grosso do Sul",           "url": "https://www.ufms.br"},
    {"sigla": "UFMT",  "nome": "Universidade Federal de Mato Grosso",                  "url": "https://www.ufmt.br"},
    {"sigla": "UFG",   "nome": "Universidade Federal de Goias",                        "url": "https://www.ufg.br"},
    {"sigla": "UFCat", "nome": "Universidade Federal de Catalao",                      "url": "https://www.ufcat.edu.br"},
    {"sigla": "UFVJM", "nome": "Universidade Federal dos Vales do Jequitinhonha e Mucuri", "url": "https://www.ufvjm.edu.br"},
    {"sigla": "UnB",   "nome": "Universidade de Brasilia",                             "url": "https://www.unb.br"},

    # Sudeste
    {"sigla": "UFES",  "nome": "Universidade Federal do Espirito Santo",               "url": "https://www.ufes.br"},
    {"sigla": "UNIFAL","nome": "Universidade Federal de Alfenas",                      "url": "https://www.unifal-mg.edu.br"},
    {"sigla": "UNIFEI","nome": "Universidade Federal de Itajuba",                      "url": "https://unifei.edu.br"},
    {"sigla": "UFJF",  "nome": "Universidade Federal de Juiz de Fora",                 "url": "https://www2.ufjf.br"},
    {"sigla": "UFLA",  "nome": "Universidade Federal de Lavras",                       "url": "https://www.ufla.br"},
    {"sigla": "UFMG",  "nome": "Universidade Federal de Minas Gerais",                 "url": "https://www.ufmg.br"},
    {"sigla": "UFOP",  "nome": "Universidade Federal de Ouro Preto",                   "url": "https://www.ufop.br"},
    {"sigla": "UFSJ",  "nome": "Universidade Federal de Sao Joao del-Rei",             "url": "https://www.ufsj.edu.br"},
    {"sigla": "UFU",   "nome": "Universidade Federal de Uberlandia",                   "url": "https://ufu.br"},
    {"sigla": "UFV",   "nome": "Universidade Federal de Vicosa",                       "url": "https://www.ufv.br"},
    {"sigla": "UFTM",  "nome": "Universidade Federal do Triangulo Mineiro",            "url": "https://www.uftm.edu.br"},
    {"sigla": "UNIFESP","nome": "Universidade Federal de Sao Paulo",                   "url": "https://www.unifesp.br"},
    {"sigla": "UFABC", "nome": "Universidade Federal do ABC",                          "url": "https://www.ufabc.edu.br"},
    {"sigla": "UFSCar","nome": "Universidade Federal de Sao Carlos",                   "url": "https://www.ufscar.br"},
    {"sigla": "UNIFEI","nome": "Universidade Federal de Itajuba",                      "url": "https://unifei.edu.br"},
    {"sigla": "UFRJ",  "nome": "Universidade Federal do Rio de Janeiro",               "url": "https://ufrj.br"},
    {"sigla": "UFRRJ", "nome": "Universidade Federal Rural do Rio de Janeiro",         "url": "https://www.ufrrj.br"},
    {"sigla": "UFF",   "nome": "Universidade Federal Fluminense",                      "url": "https://www.uff.br"},
    {"sigla": "UNIRIO","nome": "Universidade Federal do Estado do Rio de Janeiro",     "url": "https://www.unirio.br"},

    # Sul
    {"sigla": "UFPR",  "nome": "Universidade Federal do Parana",                       "url": "https://www.ufpr.br"},
    {"sigla": "UTFPR", "nome": "Universidade Tecnologica Federal do Parana",           "url": "https://www.utfpr.edu.br"},
    {"sigla": "UNIOESTE","nome": "Universidade Estadual do Oeste do Parana",           "url": "https://www.unioeste.br"},
    {"sigla": "UFFS",  "nome": "Universidade Federal da Fronteira Sul",                "url": "https://www.uffs.edu.br"},
    {"sigla": "UFSC",  "nome": "Universidade Federal de Santa Catarina",               "url": "https://ufsc.br"},
    {"sigla": "FURG",  "nome": "Universidade Federal do Rio Grande",                   "url": "https://www.furg.br"},
    {"sigla": "UFPEL", "nome": "Universidade Federal de Pelotas",                      "url": "https://portal.ufpel.edu.br"},
    {"sigla": "UFSM",  "nome": "Universidade Federal de Santa Maria",                  "url": "https://www.ufsm.br"},
    {"sigla": "UFRGS", "nome": "Universidade Federal do Rio Grande do Sul",            "url": "https://www.ufrgs.br"},
    {"sigla": "UNIPAMPA","nome": "Universidade Federal do Pampa",                      "url": "https://www.unipampa.edu.br"},
    {"sigla": "UFCSPA","nome": "Universidade Federal de Ciencias da Saude de Porto Alegre", "url": "https://www.ufcspa.edu.br"},
]

# Remover duplicatas por sigla
seen = set()
UNIVERSIDADES_UNICAS = []
for u in UNIVERSIDADES:
    if u['sigla'] not in seen:
        seen.add(u['sigla'])
        UNIVERSIDADES_UNICAS.append(u)

print(f"Total de universidades: {len(UNIVERSIDADES_UNICAS)}")

---
## 3. Configuracoes de Busca

In [4]:
# Termos de busca para documentos de regulamentacao de IA
TERMOS_IA = [
    "inteligencia artificial",
    "inteligência artificial",
    "IA generativa",
    "ia",
    "regulamentacao ia",
    "politica ia",
    "uso de ia",
    "chatgpt",
    "llm",
    "norma ia",
    "resolucao ia",
]

# Padroes de URL que tipicamente hospedam documentos normativos
URL_PADROES_NORMAS = [
    "/conselho", "/consu", "/conepe", "/cepe",
    "/resolucao", "/normativa", "/portaria",
    "/regulamento", "/deliberacao",
    "/noticias", "/news", "/atos-normativos",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; ResearchBot/1.0; +https://pesquisa-regulamentacao-ia.br)",
    "Accept-Language": "pt-BR,pt;q=0.9",
}

TIMEOUT = 15       # segundos
DELAY   = 1.5      # segundos entre requisicoes (evitar sobrecarga nos servidores)

---
## 4. Funcoes Auxiliares

In [5]:
def fetch_page(url: str) -> BeautifulSoup | None:
    """Faz GET em uma URL e retorna o objeto BeautifulSoup, ou None em caso de erro."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        r.raise_for_status()
        return BeautifulSoup(r.text, "lxml")
    except Exception:
        return None


def contem_termos_ia(texto: str) -> bool:
    """Verifica se o texto contem algum dos termos de regulamentacao de IA."""
    texto_lower = texto.lower()
    return any(t in texto_lower for t in TERMOS_IA)


def extrair_links_relevantes(soup: BeautifulSoup, base_url: str) -> list[str]:
    """Extrai links cujos textos ou hrefs contenham termos de IA ou padroes normativos."""
    links = []
    for a in soup.find_all("a", href=True):
        href  = a["href"].strip()
        texto = a.get_text(" ", strip=True)
        url   = urljoin(base_url, href)

        eh_mesmo_dominio = urlparse(url).netloc == urlparse(base_url).netloc
        tem_termo_ia     = contem_termos_ia(texto) or contem_termos_ia(href)
        tem_padrao_norma = any(p in href.lower() for p in URL_PADROES_NORMAS)

        if eh_mesmo_dominio and (tem_termo_ia or tem_padrao_norma):
            links.append(url)

    return list(set(links))


def extrair_documentos_pdf(soup: BeautifulSoup, base_url: str) -> list[str]:
    """Retorna URLs de PDFs cujos textos-ancora ou hrefs contenham termos de IA."""
    pdfs = []
    for a in soup.find_all("a", href=True):
        href  = a["href"].strip()
        texto = a.get_text(" ", strip=True)
        if href.lower().endswith(".pdf") and contem_termos_ia(texto + " " + href):
            pdfs.append(urljoin(base_url, href))
    return list(set(pdfs))


def extrair_trechos(soup: BeautifulSoup) -> list[str]:
    """Extrai trechos de texto da pagina que mencionam IA."""
    trechos = []
    for tag in soup.find_all(["p", "li", "h1", "h2", "h3", "td"]):
        texto = tag.get_text(" ", strip=True)
        if contem_termos_ia(texto) and len(texto) > 30:
            trechos.append(texto[:300])
    return trechos[:5]  # limita a 5 trechos por pagina

---
## 5. Buscador por Universidade

In [6]:
def buscar_regulamentacao(universidade: dict) -> list[dict]:
    """
    Fluxo principal de busca para uma universidade.
    1. Acessa a pagina inicial.
    2. Coleta links relevantes (conselhos, resolucoes, noticias).
    3. Visita cada link e verifica presenca de termos de IA.
    4. Registra paginas relevantes e PDFs encontrados.
    """
    resultados = []
    sigla = universidade["sigla"]
    base  = universidade["url"]

    # Pagina inicial
    soup_home = fetch_page(base)
    if soup_home is None:
        return [{"sigla": sigla, "nome": universidade["nome"],
                 "url_base": base, "url_encontrada": None,
                 "tipo": "erro", "descricao": "Site inacessivel",
                 "pdfs": "", "trecho": ""}]

    # Verifica se a propria home menciona IA
    if contem_termos_ia(soup_home.get_text()):
        trechos = extrair_trechos(soup_home)
        pdfs    = extrair_documentos_pdf(soup_home, base)
        resultados.append({
            "sigla": sigla, "nome": universidade["nome"],
            "url_base": base, "url_encontrada": base,
            "tipo": "pagina_home",
            "descricao": "Pagina inicial menciona regulamentacao de IA",
            "pdfs": " | ".join(pdfs),
            "trecho": " // ".join(trechos)
        })

    # Coleta links de 2o nivel
    links = extrair_links_relevantes(soup_home, base)
    time.sleep(DELAY)

    for link in links[:20]:  # visita no maximo 20 links por universidade
        soup_link = fetch_page(link)
        if soup_link is None:
            continue

        texto_pagina = soup_link.get_text()
        if contem_termos_ia(texto_pagina):
            trechos = extrair_trechos(soup_link)
            pdfs    = extrair_documentos_pdf(soup_link, link)
            resultados.append({
                "sigla": sigla, "nome": universidade["nome"],
                "url_base": base, "url_encontrada": link,
                "tipo": "pagina_interna",
                "descricao": "Pagina interna menciona regulamentacao de IA",
                "pdfs": " | ".join(pdfs),
                "trecho": " // ".join(trechos)
            })
        time.sleep(DELAY)

    if not resultados:
        resultados.append({
            "sigla": sigla, "nome": universidade["nome"],
            "url_base": base, "url_encontrada": None,
            "tipo": "nao_encontrado",
            "descricao": "Nenhum documento de regulamentacao de IA localizado",
            "pdfs": "", "trecho": ""
        })

    return resultados

---
## 6. Execucao Principal

In [ ]:
todos_resultados = []

for univ in tqdm(UNIVERSIDADES_UNICAS, desc="Universidades"):
    resultados = buscar_regulamentacao(univ)
    todos_resultados.extend(resultados)

df = pd.DataFrame(todos_resultados)
print(f"\nRegistros coletados: {len(df)}")
df.head()

---
## 7. Analise dos Resultados

In [ ]:
resumo = df.groupby("tipo")["sigla"].count().rename("total").reset_index()
print("Distribuicao por tipo de resultado:")
print(resumo.to_string(index=False))

In [ ]:
# Universidades com documentos encontrados
com_docs = df[df["tipo"].isin(["pagina_home", "pagina_interna"])]
print(f"Universidades com mencao a regulamentacao de IA: {com_docs['sigla'].nunique()}")
print()

# PDFs encontrados
pdfs_encontrados = df[df["pdfs"] != ""][["sigla", "nome", "pdfs"]]
print(f"Universidades com PDFs de regulamentacao: {com_docs['sigla'].nunique()}")
com_docs

In [ ]:
# Inacessiveis
inacessiveis = df[df["tipo"] == "erro"][["sigla", "nome", "url_base"]]
print(f"Sites inacessiveis: {len(inacessiveis)}")
inacessiveis

---
## 8. Exportacao

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# CSV completo
caminho_csv = f"resultados/regulamentacao_ia_{timestamp}.csv"
df.to_csv(caminho_csv, index=False, encoding="utf-8-sig")

# Excel com abas separadas
caminho_xlsx = f"resultados/regulamentacao_ia_{timestamp}.xlsx"
with pd.ExcelWriter(caminho_xlsx, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Todos", index=False)
    com_docs.to_excel(writer, sheet_name="Com_Regulamentacao", index=False)
    pdfs_encontrados.to_excel(writer, sheet_name="PDFs", index=False)
    inacessiveis.to_excel(writer, sheet_name="Inacessiveis", index=False)

print(f"Exportado: {caminho_csv}")
print(f"Exportado: {caminho_xlsx}")

In [ ]:
# Identifica universidades sem resultado positivo
sem_resultado = df[df["tipo"] == "nao_encontrado"][["sigla", "nome", "url_base"]].drop_duplicates()

print("Queries sugeridas para busca manual ou via SerpAPI:")
for _, row in sem_resultado.iterrows():
    dominio = urlparse(row["url_base"]).netloc
    query   = f'site:{dominio} "inteligencia artificial" (resolucao OR portaria OR regulamentacao)'
    print(f"  [{row['sigla']}] {query}")